<a href="https://colab.research.google.com/github/nivethithanm/mini-claw/blob/main/CLAW_02_tool_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLAW-02 — Tool Calling: Giving Your Agent Hands

**Goal:** Implement OpenAI function calling from scratch.

By the end of this notebook you'll have:
- A `Tool` abstraction — functions the LLM can call
- A `ToolRegistry` — the skill catalog
- An `AgentLoop` — the full "think → act → observe → think" cycle
- 4 built-in tools: `get_time`, `calculator`, `web_fetch`, `remember`

> **First principles mindset:**  
> Tool calling = LLM decides WHAT to call → you execute it → LLM sees the result.  
> The LLM never executes code directly. It only emits JSON.  
> You are the execution engine.


## 0. Setup — reuse NB-01 components

In [3]:
import os, json, math, datetime, inspect, urllib.request
from typing import Callable, Any
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# ---- minimal copy of NB-01 ----
from dataclasses import dataclass, field
from typing import Literal

Role = Literal["system", "user", "assistant", "tool"]

@dataclass
class Message:
    role: Role
    content: str
    tool_call_id: str = None
    tool_calls: list = None

    def to_dict(self) -> dict:
        d = {"role": self.role, "content": self.content}
        if self.tool_call_id:
            d["tool_call_id"] = self.tool_call_id
        if self.tool_calls:
            d["tool_calls"] = self.tool_calls
        return d

def build_messages(system: str, history: list[Message]) -> list[dict]:
    return [{"role": "system", "content": system}] + [m.to_dict() for m in history]

print("Setup complete ✓")


Setup complete ✓


## 1. The Tool Abstraction

A tool is: a Python function + a JSON Schema that describes it to the LLM.

The LLM sees the schema. When it wants to call a tool, it emits JSON.
You parse that JSON and call the actual Python function.

```
Tool:
  name: "calculator"
  description: "Evaluate a math expression"
  parameters: {"expression": "string"}
  fn: lambda expr: str(eval(expr))
```


In [4]:
@dataclass
class Tool:
    """
    Wraps a Python callable so the LLM can call it.

    name: unique identifier (snake_case)
    description: what the tool does — the LLM reads this
    parameters: JSON Schema for the function's arguments
    fn: the actual Python function to call
    """
    name: str
    description: str
    parameters: dict   # JSON Schema "properties" object
    fn: Callable
    required: list[str] = field(default_factory=list)

    def to_openai_spec(self) -> dict:
        """
        Convert to the format OpenAI expects in the `tools` parameter.

        Shape:
        {
            "type": "function",
            "function": {
                "name": ...,
                "description": ...,
                "parameters": {
                    "type": "object",
                    "properties": {...},
                    "required": [...]
                }
            }
        }

        Exercise: implement this.
        """
        raise NotImplementedError

    def execute(self, **kwargs) -> str:
        """
        Call the underlying function with parsed arguments.
        Always returns a string (tool results are text).
        """
        result = self.fn(**kwargs)
        return str(result)


In [5]:
# --- REFERENCE ---
@dataclass
class Tool:
    name: str
    description: str
    parameters: dict
    fn: Callable
    required: list[str] = field(default_factory=list)

    def to_openai_spec(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": self.parameters,
                    "required": self.required,
                }
            }
        }

    def execute(self, **kwargs) -> str:
        try:
            result = self.fn(**kwargs)
            return str(result)
        except Exception as e:
            return f"Error: {e}"

# Quick test
calc = Tool(
    name="calculator",
    description="Evaluate a math expression. Use Python syntax.",
    parameters={"expression": {"type": "string", "description": "Python math expression, e.g. '2 ** 10'"}},
    required=["expression"],
    fn=lambda expression: eval(expression, {"__builtins__": {}}, {"math": math}),
)
assert calc.execute(expression="2 ** 10") == "1024"
spec = calc.to_openai_spec()
assert spec["type"] == "function"
assert spec["function"]["name"] == "calculator"
print("Tool tests pass ✓")
print(json.dumps(spec, indent=2))


Tool tests pass ✓
{
  "type": "function",
  "function": {
    "name": "calculator",
    "description": "Evaluate a math expression. Use Python syntax.",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "Python math expression, e.g. '2 ** 10'"
        }
      },
      "required": [
        "expression"
      ]
    }
  }
}


## 2. The ToolRegistry

A dictionary of tools. The agent consults it to:
1. Give the LLM the list of available tools (as specs)
2. Dispatch a tool call by name when the LLM requests one


In [7]:
class ToolRegistry:
    """
    Catalog of available tools.

    The agent holds one registry and passes tool specs to every LLM call.
    """
    def __init__(self):
        self._tools: dict[str, Tool] = {}

    def register(self, tool: Tool):
        """Add a tool to the registry."""
        # TODO
        self._tools[tool.name] = tool

    def dispatch(self, name: str, arguments: str | dict) -> str:
        """
        Execute a tool by name with JSON-encoded arguments.

        arguments may be a JSON string (from OpenAI) or already a dict.
        Return the result as a string.
        If the tool doesn't exist, return an error string.
        """
        if name not in self._tools:
            return f"Error: unknown tool '{name}'"
        if isinstance(arguments, str):
            try:
                kwargs = json.loads(arguments)
            except json.JSONDecodeError as e:
                return f"Error: invalid JSON arguments: {e}"
        else:
            kwargs = arguments
        return self._tools[name].execute(**kwargs)

    def to_openai_specs(self) -> list[dict]:
        """Return all tool specs for the OpenAI API `tools` parameter."""
        return [t.to_openai_spec() for t in self._tools.values()]

    def __repr__(self):
        return f"ToolRegistry({list(self._tools.keys())})"

# Test
reg = ToolRegistry()
reg.register(calc)
print(reg)
result = reg.dispatch("calculator", '{"expression": "math.sqrt(144)"}')
print(f"sqrt(144) = {result}")


ToolRegistry(['calculator'])
sqrt(144) = 12.0


## 3. Built-in Tools

Now build the tools OpenClaw-style: each one is a small, focused function.


In [13]:
import math, datetime, urllib.request, urllib.error

# --- Tool 1: Current Time ---
def _get_time(timezone: str = "UTC") -> str:
    """Exercise: return current datetime as a readable string."""
    # Hint: datetime.datetime.utcnow()
    now = datetime.datetime.now(datetime.UTC)
    return f"{now.strftime('%Y-%m-%d %H:%M:%S')} UTC"

tool_get_time = Tool(
    name="get_time",
    description="Get the current date and time.",
    parameters={
        "timezone": {"type": "string", "description": "Timezone label (informational only)", "default": "UTC"}
    },
    required=[],
    fn=_get_time,
)


# --- Tool 2: Calculator ---
def _calculator(expression: str) -> str:
    """Exercise: safely evaluate a math expression."""
    # Use eval with a restricted namespace (no builtins, only math)
    safe_globals = {"__builtins__": {}, "math": math} # Add math to safe_globals
    safe_locals = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    try:
        result = eval(expression, safe_globals, safe_locals)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

tool_calculator = Tool(
    name="calculator",
    description="Evaluate a mathematical expression. Supports Python math syntax and the math module (e.g. math.sqrt, math.pi).",
    parameters={"expression": {"type": "string", "description": "Python math expression"}},
    required=["expression"],
    fn=_calculator,
)


# --- Tool 3: Web Fetch (lightweight) ---
def _web_fetch(url: str, max_chars: int = 2000) -> str:
    """Exercise: fetch a URL and return the first max_chars characters of the response."""
    url = url.strip()
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "MiniClaw/1.0"})
        with urllib.request.urlopen(req, timeout=5) as resp:
            content = resp.read().decode("utf-8", errors="ignore")
            # Strip HTML tags roughly
            import re
            text = re.sub(r'<[^>]+>', ' ', content)
            text = re.sub(r'\s+', ' ', text).strip()
            return text
    except Exception as e:
        return f"Error fetching {url}: {e}"

tool_web_fetch = Tool(
    name="web_fetch",
    description="Fetch the text content of a URL. Returns up to 2000 characters of the page content.",
    parameters={
        "url": {"type": "string", "description": "The URL to fetch"},
        "max_chars": {"type": "integer", "description": "Max characters to return", "default": 2000},
    },
    required=["url"],
    fn=_web_fetch,
)


# --- Tool 4: Memory Write ---
_memory_store: dict = {}

def _remember(key: str, value: str) -> str:
    """Exercise: store key/value in the in-memory store. Return confirmation."""
    _memory_store[key] = value
    return f"Stored '{key}'."

def _recall(key: str) -> str:
    """Return the value for a key, or 'Not found'."""
    return _memory_store.get(key, f"No memory found for '{key}'.")

tool_remember = Tool(
    name="remember",
    description="Store a piece of information under a key for later recall.",
    parameters={
        "key": {"type": "string", "description": "The key to store under"},
        "value": {"type": "string", "description": "The information to store"},
    },
    required=["key", "value"],
    fn=_remember,
)

tool_recall = Tool(
    name="recall",
    description="Retrieve a stored memory by key.",
    parameters={"key": {"type": "string", "description": "The key to look up"}},
    required=["key"],
    fn=_recall,
)

# Tests
print(_get_time())
print(_calculator("math.sqrt(2) ** 2"))
print(_remember("user_name", "Niv"))
print(_recall("user_name"))
print(_recall("missing_key"))

2026-05-29 05:02:13 UTC
2.0000000000000004
Stored 'user_name'.
Niv
No memory found for 'missing_key'.


## 4. The Agent Loop

This is the core of everything. The loop:

```
User message
    ↓
LLM call (with tools)
    ↓
LLM returns text? → done
LLM returns tool_call? →
    Execute tool
    Add tool_result to history
    Loop back to LLM call
```

The LLM may chain multiple tool calls. Loop until it returns plain text.


In [15]:
def agent_loop(
    user_message: str,
    system_prompt: str,
    registry: ToolRegistry,
    history: list[Message],
    model: str = "gpt-4o-mini",
    max_tool_calls: int = 5,
    verbose: bool = True,
) -> tuple[str, list[Message]]:
    """
    Run one full agent turn (may include multiple tool calls).

    Returns:
        (final_text_reply, updated_history)

    Exercise:
    1. Add user_message to history
    2. Call the LLM with tool specs
    3. If response has tool_calls:
       a. Add assistant message (with tool_calls field) to history
       b. For each tool call: dispatch, add tool result message
       c. Go to step 2
    4. If response is plain text: add to history, return
    5. Stop after max_tool_calls iterations (safety)
    """
    history = list(history)
    history.append(Message(role="user", content=user_message))
    tools_spec = registry.to_openai_specs()
    call_count = 0

    while call_count < max_tool_calls:
        messages = build_messages(system_prompt, history)
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools_spec if tools_spec else None,
            tool_choice="auto" if tools_spec else None,
        )
        msg = response.choices[0].message

        # Case 1: Tool call
        if msg.tool_calls:
            call_count += 1
            history.append(
                Message(
                    role="assistant",
                    content=msg.content or "",
                    tool_calls=[tc.model_dump() for tc in msg.tool_calls],
                )
            )
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = tc.function.arguments
                if verbose:
                    print(f"  🔧 Calling tool: {fn_name}({fn_args})")
                result = registry.dispatch(fn_name, fn_args)

                if verbose:
                    print(f"  📦 Result: {result[:100]}...")
                history.append(Message(
                    role="tool",
                    content=result,
                    tool_call_id=tc.id,
                ))

        # Case 2: No Tool call
        else:
            final_reply = msg.content or ""
            history.append(Message(role="assistant", content=final_reply))
            return final_reply, history

    return "Max tool calls reached.", history

In [16]:
# Test the full loop
reg = ToolRegistry()
for t in [tool_get_time, tool_calculator, tool_remember, tool_recall, tool_web_fetch]:
    reg.register(t)

SYSTEM = """You are Claw, a sharp personal assistant.
Use tools when needed. Be concise."""

history = []

print("=== Turn 1: math ===")
reply, history = agent_loop(
    "What is the square root of 2 to 6 decimal places?",
    SYSTEM, reg, history, verbose=True
)
print(f"Reply: {reply}\n")

print("=== Turn 2: time + memory ===")
reply, history = agent_loop(
    "What time is it? Also, remember that I prefer dark mode.",
    SYSTEM, reg, history, verbose=True
)
print(f"Reply: {reply}\n")

print("=== Turn 3: recall ===")
reply, history = agent_loop(
    "What display preference did I mention?",
    SYSTEM, reg, history, verbose=True
)
print(f"Reply: {reply}\n")


=== Turn 1: math ===
  🔧 Calling tool: calculator({"expression":"math.sqrt(2)"})
  📦 Result: 1.4142135623730951...
Reply: The square root of 2 to six decimal places is 1.414214.

=== Turn 2: time + memory ===
  🔧 Calling tool: get_time({})
  📦 Result: 2026-05-29 05:09:44 UTC...
  🔧 Calling tool: remember({"key": "preference", "value": "dark mode"})
  📦 Result: Stored 'preference'....
Reply: The current time is 05:09:44 UTC. Your preference for dark mode has been noted.

=== Turn 3: recall ===
  🔧 Calling tool: recall({"key":"preference"})
  📦 Result: dark mode...
Reply: You mentioned a preference for dark mode.



## 5. Putting It Together: ToolAgent

Wrap `agent_loop` into a stateful class that mirrors `ChatSession` from NB-01.


In [21]:
class ToolAgent:
    """
    A ChatSession with tools.

    Exercise: implement __init__ and chat().
    chat() should call agent_loop and update self.history.
    """
    def __init__(self, system_prompt: str, registry: ToolRegistry, model: str = "gpt-4o-mini"):
        self.system_prompt = system_prompt
        self.registry = registry
        self.model = model
        self.history: list[Message] = []

    def chat(self, user_message: str, verbose: bool = True) -> str:
        # TODO: call agent_loop, update self.history, return reply
        reply, self.history = agent_loop(
            user_message, self.system_prompt, self.registry,
            self.history, self.model, verbose=verbose
        )
        return reply

    def reset(self):
        self.history.clear()

    @property
    def turn_count(self) -> int:
        return sum(1 for m in self.history if m.role == "user")


# Full demo
agent = ToolAgent(SYSTEM, reg)
print(agent.chat("What's 15% of 847?", verbose=False))
print(agent.chat("Remember my favorite number is 42.", verbose=False))
print(agent.chat("What's my favorite number?", verbose=False))
print(f"Total turns: {agent.turn_count}")

15% of 847 is 127.05.
I've remembered your favorite number as 42.
Your favorite number is 42.
Total turns: 3


## 6. Exercises

**E1.** Add a `run_shell` tool that executes a shell command and returns stdout.  
Add a safety check: only allow commands from an allowlist (`["ls", "pwd", "date", "echo"]`).

**E2.** Implement `tool_choice="required"` mode — force the agent to always use a tool on the first call. When would this be useful?

**E3.** Add a `max_retries` parameter to `agent_loop` that catches `ToolDispatchError` and retries that specific tool call with a different argument (hint: add the error to the next user message as context).

**E4 (hard):** Implement parallel tool calling — when the LLM returns multiple tool calls in one response, execute them concurrently using `ThreadPoolExecutor`, then batch the results back.

---

## ✅ Checkpoint

You now have:
- `Tool`: Python function + JSON schema
- `ToolRegistry`: catalog + dispatcher  
- `agent_loop()`: the think → act → observe cycle
- `ToolAgent`: stateful agent with tools

**Next:** CLAW-03 — Memory. Your agent remembers across sessions.
